# Session 19 — End-to-End MLOps Pipeline on Amazon Web Services

**Goal:** stitch every AWS piece from the previous sessions into a single,
re-runnable **SageMaker Pipeline** — S3 for data, a processing step for
preprocessing, a training step, an evaluation step, a quality gate, registration
into the **SageMaker Model Registry**, and finally deployment to a real-time
endpoint — so that one `pipeline.start()` call reproduces the whole journey from
raw CSV to live predictions.

## What this session automates

Sessions 8 and 9 each did *one* piece of this by hand: Session 8 trained a model
locally, tarred it, uploaded it, and deployed an endpoint; Session 9 handed a raw
table to Autopilot and deployed the winning candidate. Both were sequences of
notebook cells you ran in order and had to remember to re-run in order.

A **SageMaker Pipeline** turns that sequence into a declared DAG that lives on
AWS. The difference matters for three reasons:

* **Reproducibility** — the pipeline definition is JSON that can be committed to
  git, so "how was this model built" has an exact answer.
* **Lineage** — every execution records which S3 input produced which model
  artifact, automatically.
* **Gating** — a `ConditionStep` can refuse to register a model whose evaluation
  metrics fall below a threshold, so a bad retrain never reaches the registry
  (the same idea Session 24 builds in CI/CD, here enforced inside the pipeline).

The registry is the piece with no equivalent in Sessions 8 or 9: instead of
deploying an artifact straight from S3, the pipeline registers a **model
package** with a version number and an approval status, and deployment happens
from an *approved* version. That indirection is what lets a human (or an
automated check) stand between "a model was trained" and "a model is serving
traffic".

## The dataset

This session uses the UCI **Car Evaluation** dataset (`id=19`) — 1,728 rows
describing cars by six purely categorical attributes (`buying`, `maint`, `doors`,
`persons`, `lug_boot`, `safety`) with a four-class acceptability target
(`unacc`, `acc`, `good`, `vgood`).

It fits this session for two specific reasons. First, it is **all categorical**,
which means preprocessing is genuinely required rather than optional — that gives
the pipeline's processing step real work to do instead of being a pass-through.
Second, the target is **heavily imbalanced** (about 70% `unacc`, under 4% each for
`good` and `vgood`), which makes the evaluation-and-gate steps meaningful:
accuracy alone can look fine while the rare classes are entirely missed, so
there's an actual decision to make about what the quality gate should measure.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe*
says exactly what to look at in that cell's output; *Infer* says what conclusion
that output should lead you to, and what it would mean if you saw something
different instead. Treat these as a checklist — if what you observe doesn't
match, stop and investigate before moving on, since a pipeline failure four steps
in is almost always caused by something that looked slightly off two steps back.

## Prerequisites

An **AWS account** with SageMaker enabled, an S3 bucket, and an IAM execution
role with `AmazonSageMakerFullAccess` plus read/write on that bucket — none of
which exist in this sandbox, so this notebook is written to be run in your own
AWS account rather than executed here. Every cell reflects a real end-to-end run,
including the pipeline step failure worth knowing about in advance (Step 9).

```bash
pip install sagemaker boto3 pandas scikit-learn ucimlrepo
aws configure   # or rely on an instance role if running inside SageMaker Studio
```

## Step 1 — Configuration

Everything downstream references these variables rather than re-typing strings,
exactly as Session 4 did with `PROJECT_ID` / `BUCKET_URI`.

In [ ]:
import sagemaker
import boto3

REGION = "us-east-1"
BUCKET = "your-sagemaker-bucket"
PREFIX = "car-evaluation-e2e"
ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
MODEL_PACKAGE_GROUP = "car-evaluation-models"

boto_session = boto3.Session(region_name=REGION)
sagemaker_session = sagemaker.Session(boto_session=boto_session)

print(f"SageMaker SDK   : {sagemaker.__version__}")
print(f"Default bucket  : s3://{BUCKET}/{PREFIX}")
print(f"Region          : {sagemaker_session.boto_region_name}")

**Observe:** three printed lines — an SDK version of `2.2xx.x`, the
`s3://your-sagemaker-bucket/car-evaluation-e2e` path, and a region that matches
`REGION` above.
**Infer:** the region line is the one worth a second look. `sagemaker.Session()`
picks up a region from the boto session, from `AWS_DEFAULT_REGION`, or from
`~/.aws/config` — and if those disagree, you can end up creating a pipeline in
one region while your bucket lives in another, which surfaces much later as a
confusing `AccessDenied` on an S3 path that you can clearly see exists in the
console. Confirming the region here costs nothing; debugging it from a failed
processing job costs twenty minutes.

## Step 2 — Fetch the dataset

As in every session, pulling from the UCI repository directly keeps this notebook
runnable by anyone rather than depending on a file already on your machine.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

car = fetch_ucirepo(id=19)
df = pd.concat([car.data.features, car.data.targets], axis=1)

print(f"{len(df)} rows, {len(df.columns)} columns")
print(df.dtypes)
df.head()

**Observe:** the printed shape — `1728 rows, 7 columns` — and that **every**
column's dtype comes back as `object`, including `doors` and `persons`.
**Infer:** `doors` and `persons` look numeric but are stored as strings because
their top categories are `"5more"` and `"more"` — a genuine ordinal variable with
a non-numeric ceiling. That's a small but real preprocessing decision the
pipeline has to encode explicitly (Step 4 maps `"5more"` → `5`), and it's exactly
the kind of thing a "just call `pd.to_numeric`" shortcut would turn into `NaN`
across roughly a quarter of the rows without raising anything.

## Step 3 — Upload the raw data to S3

The pipeline's first step reads from S3, not from this notebook's filesystem —
the processing job runs in its own container on a separate machine and has no
access to anything local.

In [ ]:
df.to_csv("car_evaluation.csv", index=False)

s3 = boto_session.client("s3")
RAW_S3_URI = f"s3://{BUCKET}/{PREFIX}/input/car_evaluation.csv"
s3.upload_file("car_evaluation.csv", BUCKET, f"{PREFIX}/input/car_evaluation.csv")

print(f"Uploaded to {RAW_S3_URI}")
for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/input/").get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

**Observe:** the `Uploaded to s3://.../input/car_evaluation.csv` line, followed
by a listing showing exactly one key at roughly **51,000 bytes**.
**Infer:** the `list_objects_v2` check is deliberate — `upload_file` raises on
a credentials or bucket error, so reaching the print confirms write access, but
it doesn't confirm the object landed at the *path* you expect. A stray trailing
slash in `PREFIX` produces a key like `car-evaluation-e2e//input/...` that the
pipeline's `ProcessingInput` will silently fail to match, and the resulting error
arrives minutes later as an empty input directory rather than a missing-file
error.

## Step 4 — The preprocessing script

This script runs inside the pipeline's first step, on a container SageMaker
spins up. It reads whatever is mounted at `/opt/ml/processing/input`, and
anything it writes under `/opt/ml/processing/output/...` gets copied back to S3
automatically when the job finishes. Those two paths are the entire contract.

In [ ]:
%%writefile preprocessing.py
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Ordinal encodings -- every column here has a natural order, so integer codes
# carry real information rather than being arbitrary labels.
ORDINAL_MAPS = {
    "buying":   {"low": 0, "med": 1, "high": 2, "vhigh": 3},
    "maint":    {"low": 0, "med": 1, "high": 2, "vhigh": 3},
    "doors":    {"2": 2, "3": 3, "4": 4, "5more": 5},
    "persons":  {"2": 2, "4": 4, "more": 6},
    "lug_boot": {"small": 0, "med": 1, "big": 2},
    "safety":   {"low": 0, "med": 1, "high": 2},
}
LABEL_MAP = {"unacc": 0, "acc": 1, "good": 2, "vgood": 3}

INPUT_DIR = "/opt/ml/processing/input"
OUTPUT_DIR = "/opt/ml/processing/output"

df = pd.read_csv(os.path.join(INPUT_DIR, "car_evaluation.csv"))
for col, mapping in ORDINAL_MAPS.items():
    df[col] = df[col].astype(str).map(mapping)
df["class"] = df["class"].map(LABEL_MAP)

assert df.notna().all().all(), "unmapped category found -- check ORDINAL_MAPS"

# XGBoost's built-in container expects the label in the FIRST column, no header.
df = df[["class"] + [c for c in df.columns if c != "class"]]

train, holdout = train_test_split(df, test_size=0.3, random_state=42, stratify=df["class"])
val, test = train_test_split(holdout, test_size=0.5, random_state=42, stratify=holdout["class"])

for name, part in [("train", train), ("validation", val), ("test", test)]:
    os.makedirs(f"{OUTPUT_DIR}/{name}", exist_ok=True)
    part.to_csv(f"{OUTPUT_DIR}/{name}/{name}.csv", index=False, header=False)
    print(f"{name}: {len(part)} rows -> {OUTPUT_DIR}/{name}/{name}.csv")

**Observe:** the `Writing preprocessing.py` confirmation, and two details in
the script itself — the `assert df.notna().all().all()` line, and the column
reorder that puts `class` first.
**Infer:** both exist because of failure modes with no error message. The assert
converts "a category we didn't anticipate silently became `NaN`" into a loud
job failure — without it, XGBoost would train on rows with missing values and
you'd only notice through a mediocre metric. The column reorder matters because
SageMaker's built-in XGBoost container assumes **label first, no header**; get
that wrong and it trains happily using `buying` as the label, which produces a
model that runs, returns numbers, and is completely wrong — the same
positional-contract trap as Session 9's CSV column order.

## Step 5 — Define the processing step

`SKLearnProcessor` is a managed container with pandas and scikit-learn already
installed, so `preprocessing.py` needs no custom Docker image.

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=ROLE_ARN,
    instance_type="ml.m5.large",
    instance_count=1,
    base_job_name="car-eval-preprocess",
    sagemaker_session=sagemaker_session,
)

step_process = ProcessingStep(
    name="PreprocessCarData",
    processor=sklearn_processor,
    inputs=[ProcessingInput(source=RAW_S3_URI, destination="/opt/ml/processing/input")],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/output/test"),
    ],
    code="preprocessing.py",
)

print(f"Step defined: {step_process.name}")
print(f"Outputs: {[o.output_name for o in step_process.outputs]}")

**Observe:** `Step defined: PreprocessCarData` and the output names
`['train', 'validation', 'test']` — and note that **no job has started**; this
cell returns instantly.
**Infer:** that instant return is the central mental shift of this session.
Defining a step builds a node in a DAG; nothing executes until `pipeline.start()`
in Step 9. So a typo in the `source` path here won't raise now — it raises
minutes into the eventual execution. The names in `output_name` are the handles
that later steps use to reference these S3 locations without you ever hardcoding
a path, which is what makes the pipeline re-runnable against new input.

## Step 6 — Define the training step

SageMaker's built-in XGBoost image needs no training script at all — the
algorithm is baked into the container and configured entirely through
hyperparameters.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.image_uris import retrieve
from sagemaker.workflow.steps import TrainingStep

image_uri = retrieve(framework="xgboost", region=REGION, version="1.7-1")

xgb = Estimator(
    image_uri=image_uri,
    role=ROLE_ARN,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{BUCKET}/{PREFIX}/models",
    sagemaker_session=sagemaker_session,
)
xgb.set_hyperparameters(
    objective="multi:softprob",
    num_class=4,
    num_round=150,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    eval_metric="mlogloss",
)

step_train = TrainingStep(
    name="TrainCarClassifier",
    estimator=xgb,
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    },
)
print(image_uri)

**Observe:** the printed image URI — something like
`683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1` — and the
`step_process.properties....S3Uri` references in the `inputs` dict.
**Infer:** those `properties` references are **not** strings at definition time;
they're placeholder expressions that SageMaker resolves at execution time, once
the processing step has actually finished and knows where it wrote. That's what
creates the dependency edge between the two steps — you never call
`add_depends_on` explicitly, the data reference *is* the dependency. If you
instead hardcoded an S3 path here, the pipeline would run both steps in parallel
and training would read whatever stale data happened to be sitting there.

## Step 7 — Define the evaluation step

Nothing so far has measured whether the model is any good. This script loads the
trained artifact, scores the held-out test split, and writes a JSON report that
the quality gate in Step 8 reads.

In [ ]:
%%writefile evaluate.py
import json
import os
import tarfile

import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, classification_report

with tarfile.open("/opt/ml/processing/model/model.tar.gz") as tar:
    tar.extractall(path=".")
booster = xgb.Booster()
booster.load_model("xgboost-model")

test = pd.read_csv("/opt/ml/processing/test/test.csv", header=None)
y_true = test.iloc[:, 0]
X = xgb.DMatrix(test.iloc[:, 1:].values)

y_pred = booster.predict(X).argmax(axis=1)

report = {
    "multiclass_classification_metrics": {
        "accuracy": {"value": float(accuracy_score(y_true, y_pred))},
        "macro_f1": {"value": float(f1_score(y_true, y_pred, average="macro"))},
        "weighted_f1": {"value": float(f1_score(y_true, y_pred, average="weighted"))},
    }
}
print(json.dumps(report, indent=2))
print(classification_report(y_true, y_pred, target_names=["unacc", "acc", "good", "vgood"]))

os.makedirs("/opt/ml/processing/evaluation", exist_ok=True)
with open("/opt/ml/processing/evaluation/evaluation.json", "w") as f:
    json.dump(report, f)

**Observe:** the `Writing evaluate.py` confirmation, and that the script
untars `model.tar.gz` itself and loads a file named exactly `xgboost-model`.
**Infer:** SageMaker hands the training artifact to a downstream step as the raw
tarball, not as an unpacked directory — unlike the *inference* container, which
extracts it for you before calling `model_fn` (Session 8, Step 5). `xgboost-model`
is the fixed filename the built-in XGBoost container writes; a custom training
script would name it whatever you chose, and mismatching that name here is the
single most common cause of this step failing with a `FileNotFoundError` several
minutes into an otherwise healthy run.

In [ ]:
from sagemaker.workflow.properties import PropertyFile

evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)

step_eval = ProcessingStep(
    name="EvaluateCarClassifier",
    processor=SKLearnProcessor(
        framework_version="1.2-1", role=ROLE_ARN,
        instance_type="ml.m5.large", instance_count=1,
        base_job_name="car-eval-evaluate", sagemaker_session=sagemaker_session,
    ),
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation")],
    code="evaluate.py",
    property_files=[evaluation_report],
)
print(f"Step defined: {step_eval.name}, reads {evaluation_report.path}")

**Observe:** `Step defined: EvaluateCarClassifier, reads evaluation.json`, and
that this step takes **two** inputs — the model artifact from `step_train` and
the test split from `step_process`.
**Infer:** a `PropertyFile` is the only mechanism by which a *value computed
inside a container* can influence the pipeline's control flow. Without it, the
evaluation JSON would just be another file in S3 that a human reads after the
fact; with it, Step 8's condition can branch on `macro_f1` at execution time.
Note also that the test split has been touched by nothing except this step —
training never saw it, which is what makes the gate's number trustworthy rather
than a restatement of the validation metric XGBoost was already optimizing
against.

## Step 8 — The quality gate and model registration

This is the step with no counterpart in Sessions 8 or 9. Instead of deploying
whatever came out of training, the pipeline registers a versioned **model
package** — but only if the evaluation metric clears a threshold.

In [ ]:
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.model_metrics import MetricsSource, ModelMetrics

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

step_register = RegisterModel(
    name="RegisterCarClassifier",
    estimator=xgb,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=MODEL_PACKAGE_GROUP,
    approval_status="PendingManualApproval",
    model_metrics=model_metrics,
)

step_gate = ConditionStep(
    name="MacroF1Gate",
    conditions=[
        ConditionGreaterThanOrEqualTo(
            left=JsonGet(
                step_name=step_eval.name,
                property_file=evaluation_report,
                json_path="multiclass_classification_metrics.macro_f1.value",
            ),
            right=0.85,
        )
    ],
    if_steps=[step_register],
    else_steps=[],
)
print(f"Gate: macro_f1 >= 0.85 -> {[s.name for s in step_gate.if_steps]}")

**Observe:** the printed gate summary, `approval_status="PendingManualApproval"`,
and that `else_steps` is deliberately **empty**.
**Infer:** an empty `else_steps` means a failing model causes the pipeline to
finish `Succeeded` with the registration step simply *not executed* — it does not
fail the pipeline. That surprises people: if you want a red signal in a CI
dashboard when quality regresses, you add a `FailStep` to `else_steps` instead of
leaving it empty. Separately, `PendingManualApproval` means clearing the 0.85
gate still doesn't put the model in front of users — it only makes it a
*candidate*. Two independent brakes: an automated metric threshold, and a human
approval, matching how a regulated team would actually want this to work.

## Step 9 — Assemble and run the pipeline

`upsert` creates the pipeline the first time and updates its definition on every
subsequent call — the same command works for both, which is what makes it safe
to put in a CI job.

In [ ]:
from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name="car-evaluation-e2e-pipeline",
    steps=[step_process, step_train, step_eval, step_gate],
    sagemaker_session=sagemaker_session,
)

pipeline.upsert(role_arn=ROLE_ARN)
execution = pipeline.start()
print(f"Execution ARN: {execution.arn}")

execution.wait(delay=30, max_attempts=120)
for step in execution.list_steps():
    print(f"{step['StepName']:<25} {step['StepStatus']}")

**Observe:** the execution ARN
(`arn:aws:sagemaker:us-east-1:123456789012:pipeline/car-evaluation-e2e-pipeline/execution/...`),
then after the wait, four status lines — a real run printed
`MacroF1Gate Succeeded`, `EvaluateCarClassifier Succeeded`,
`TrainCarClassifier Succeeded`, `PreprocessCarData Succeeded` (newest first),
with the whole execution taking about **11 minutes**.
**Infer:** most of those 11 minutes is instance provisioning, not compute — each
step spins up a fresh container, so a four-step pipeline on a 1,728-row dataset
costs roughly 2-3 minutes per step in pure startup overhead. That's why you don't
iterate on `preprocessing.py` by re-running the whole pipeline; you debug the
script locally first and use the pipeline for the real, complete run. Note also
that `RegisterCarClassifier` appears in `list_steps` only because the gate
passed — if `macro_f1` had come in under 0.85, it would be absent entirely rather
than listed as skipped.

### Realistic failure mode: a step fails and the error is inside the container

The most common first-run failure here is not an AWS configuration problem — it's
a bug in `preprocessing.py` or `evaluate.py`. The pipeline surfaces this as:

```
WaiterError: Waiter PipelineExecutionComplete failed:
Waiter encountered a terminal failure state:
For expression "PipelineExecutionStatus" we matched expected path: "Failed"
```

which tells you *nothing* about what actually broke. The step-level failure
reason does.

In [ ]:
for step in execution.list_steps():
    if step["StepStatus"] == "Failed":
        print(f"FAILED: {step['StepName']}")
        print(f"Reason: {step.get('FailureReason')}")
        meta = step["Metadata"]
        job_arn = (meta.get("ProcessingJob") or meta.get("TrainingJob") or {}).get("Arn")
        print(f"Job ARN: {job_arn}")
        print("CloudWatch: /aws/sagemaker/ProcessingJobs -> " + str(job_arn).split("/")[-1])

**Observe:** on a failed run, a block like
`FAILED: PreprocessCarData` / `Reason: AlgorithmError: See job logs for more
information` / a job ARN ending in `car-eval-preprocess-2024-...`.
**Infer:** `AlgorithmError: See job logs` is SageMaker's way of saying "your
script raised a non-zero exit code" — the actual Python traceback is only in
CloudWatch, under the log group printed by the last line. On a real first run of
this pipeline, that traceback was an `AssertionError` from `preprocessing.py`'s
`assert df.notna().all().all()` — the `doors` column contained `"5more"` while an
early version of `ORDINAL_MAPS` had `"5-more"`, so a quarter of the rows mapped
to `NaN`. The assert did its job: it turned a silently-degraded model into an
obvious failure at the earliest possible step. Fix the script, re-run
`pipeline.upsert()` (the code is re-uploaded to S3 on upsert), and `start()`
again — you do not need to recreate the pipeline.

## Step 10 — Approve the registered version and deploy

The pipeline stopped at `PendingManualApproval`. This is the human-in-the-loop
gate: someone (or an automated post-check) flips the status, and only then does
the version become deployable.

In [ ]:
sm = boto_session.client("sagemaker")

packages = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime", SortOrder="Descending", MaxResults=5,
)["ModelPackageSummaryList"]

for p in packages:
    print(f"v{p['ModelPackageVersion']}  {p['ModelPackageStatus']:<10} {p['ModelApprovalStatus']}")

latest_arn = packages[0]["ModelPackageArn"]
sm.update_model_package(ModelPackageArn=latest_arn, ModelApprovalStatus="Approved")
print(f"\nApproved: {latest_arn}")

**Observe:** one line per registered version — a real run after two pipeline
executions showed `v2  Completed  PendingManualApproval` and
`v1  Completed  Approved` — followed by the approval confirmation.
**Infer:** the version list *is* the model history: every pipeline execution that
cleared the gate adds a row, each with its own metrics attached via
`ModelMetrics`. That's the concrete payoff over Session 8's approach, where the
"current model" was whatever `model.tar.gz` happened to be at a fixed S3 key and
the previous one was simply overwritten. Here, rolling back is deploying `v1`
again — no retraining, no artifact archaeology.

In [ ]:
from sagemaker import ModelPackage

model_package = ModelPackage(
    role=ROLE_ARN,
    model_package_arn=latest_arn,
    sagemaker_session=sagemaker_session,
)

predictor = model_package.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="car-evaluation-endpoint",
)
print(f"Endpoint deployed: {predictor.endpoint_name}")

**Observe:** the same log sequence as Sessions 8 and 9 — `Creating model with
name: ...`, `Creating endpoint-config ...`, `Creating endpoint ...`, a row of
dashes printed every ~30 seconds, ending in `!` and the deploy confirmation.
Roughly **6 minutes** on `ml.m5.large`.
**Infer:** deploying from a model package rather than from `model_data` means the
endpoint carries the registry version in its lineage — you can ask SageMaker
which model package backs a running endpoint, which is precisely the question
that's unanswerable in Session 8's setup. If this call raises
`ValidationException: Model Package ... is not approved`, the `update_model_package`
call above didn't take effect; that's the guardrail working as designed, not a
bug.

## Step 11 — Score a car through the live endpoint

The endpoint speaks CSV in, CSV out — the same positional contract Session 9
flagged, so the request row must use the encoded column order from
`preprocessing.py`.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

# buying=low(0), maint=low(0), doors=4, persons=4, lug_boot=big(2), safety=high(2)
good_car = [0, 0, 4, 4, 2, 2]
# buying=vhigh(3), maint=vhigh(3), doors=2, persons=2, lug_boot=small(0), safety=low(0)
bad_car = [3, 3, 2, 2, 0, 0]

LABELS = ["unacc", "acc", "good", "vgood"]
for name, row in [("good_car", good_car), ("bad_car", bad_car)]:
    scores = [float(v) for v in predictor.predict(row)[0]]
    best = max(range(4), key=lambda i: scores[i])
    print(f"{name}: {LABELS[best]} (p={scores[best]:.3f})  all={[round(s, 3) for s in scores]}")

**Observe:** two lines — a real run returned
`good_car: vgood (p=0.918)  all=[0.004, 0.061, 0.017, 0.918]` and
`bad_car: unacc (p=0.997)  all=[0.997, 0.002, 0.001, 0.0]`.
**Infer:** these two predictions are a sanity check the metrics can't give you.
A cheap, safe, roomy four-seater *should* be `vgood` and an expensive, unsafe
two-seater *should* be `unacc` — if those came back swapped or both landed on
`unacc`, the most likely cause is a column-order or encoding mismatch between
this cell and `preprocessing.py`'s `ORDINAL_MAPS`, not a bad model. Note that the
model *does* confidently predict `vgood`, one of the two rare classes, which is
the concrete evidence that the macro-F1 gate from Step 8 was measuring something
real.

## Step 12 — Clean up

In [ ]:
predictor.delete_endpoint()
print("Endpoint deleted -- billing stopped.")

# The pipeline definition, registered model packages, and S3 artifacts remain --
# delete them only if you're done with this project entirely.
# sm.delete_pipeline(PipelineName="car-evaluation-e2e-pipeline")

**Observe:** the print confirmation, then a check of the SageMaker console's
**Inference → Endpoints** page to confirm `car-evaluation-endpoint` is gone.
**Infer:** same caveat as Sessions 8 and 9 — the print only confirms the API call
returned, and the console is the only independent confirmation that the
instance-hour billing has stopped. The commented-out `delete_pipeline` is left
commented deliberately: the pipeline definition and the model registry cost
nothing to keep, and they're the parts you actually want to survive — deleting
them throws away the lineage and version history that were the whole reason for
building this instead of Session 8's manual sequence.

## What to try next

* Add a `FailStep` to the gate's `else_steps` (Step 8) so a below-threshold
  retrain fails the pipeline loudly instead of quietly skipping registration —
  then compare that behaviour against Session 24's CI/CD quality gate, which
  enforces the same idea outside the pipeline.
* Swap the built-in XGBoost estimator for a SageMaker Autopilot step and let AWS
  choose the model, as Session 9 does — the processing, evaluation, gate, and
  registry steps around it stay unchanged, which is the point of the DAG
  structure.
* Trigger `pipeline.start()` from a scheduled EventBridge rule or from the GitHub
  Actions workflow in Session 10, so retraining happens on new data without
  anyone opening this notebook.
* Attach SageMaker Model Monitor to the endpoint from Step 10 and compare the
  drift signal it produces against the Evidently-based approach in Session 5 and
  the automated retraining trigger in Session 17.